# Calling procedures by methods

Everything amoorion can do is a method: it has `Params`, a `run`, and front-ends
built from them. The same call looks the same from everywhere.

In [ ]:
import os, tempfile
from pathlib import Path

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault), an inbox and a dataset in a temporary directory.
scratch = Path(tempfile.mkdtemp())
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = str(scratch / "inbox.sqlite3")
os.environ["AMOORION_DATASET"] = str(scratch / "dataset")

In [ ]:
from amoorion import Amoorion
from amoorion.methods import METHODS

for name in METHODS.names():
    method = METHODS.get(name)
    print(f"{name:10} {'read-only' if method.read_only else 'changes  '} {method.description}")

What a method takes is a model, so its schema is known to anything that asks:

In [ ]:
METHODS.get("publish").Params.model_json_schema()["required"]

From the object, by name or by `call`:

In [ ]:
amo = Amoorion().use("dryrun")
amo.publish(text="by its name")[0]["ok"], amo.call("publish", text="by call")[0]["ok"]

Several as one procedure: `workflow` runs them in order and stops at the first
failure. `$last` is the text of the previous answer.

In [ ]:
steps = [
    {"method": "publish", "params": {"text": "the news", "to": ["dryrun"]}},
    {"method": "publish", "params": {"text": "repeated: $last", "to": ["dryrun"]}},
    {"method": "publish", "params": {"text": "this fails", "to": ["nowhere"]}},
    {"method": "publish", "params": {"text": "never runs", "to": ["dryrun"]}},
]
for step in amo.workflow(steps=steps):
    print(step["method"], step["ok"], step["error"] or step["result"][0]["message"]["text"])